# FleetManager — Analyse de la flotte

**Auteur** : Raphaël Quilichini  
**Module** : Outils technologiques pour la digitalisation (ESSCA)  
**Contexte métier** : pilotage du TCO (Total Cost of Ownership) d'une flotte automobile.

Ce notebook répond à la phase **Visualisation (07/05)** : agrégations, regroupements, extrêmes, comptages, accompagnés d'une analyse métier.

In [6]:
/**
 * Représente une opération d'entretien effectuée sur un véhicule
 * (ex : pneus à changer tous les 40 000 km pour 600 €).
 */
public class Entretien {

    private String idEntretien;
    private String idVehicule;
    private String idTypeEntretien;
    private double coutUnitaire;   // coût en € d'une opération
    private int frequenceKm;       // tous les combien de km

    /**
     * Constructeur classique.
     * @param idEntretien identifiant unique de l'entretien
     * @param idVehicule identifiant du véhicule concerné
     * @param idTypeEntretien identifiant du type d'entretien
     * @param coutUnitaire coût en euros d'une opération
     * @param frequenceKm fréquence de l'entretien en kilomètres
     */
    public Entretien(String idEntretien, String idVehicule, String idTypeEntretien,
                     double coutUnitaire, int frequenceKm) {
        setIdEntretien(idEntretien);
        setIdVehicule(idVehicule);
        setIdTypeEntretien(idTypeEntretien);
        setCoutUnitaire(coutUnitaire);
        setFrequenceKm(frequenceKm);
    }

    /**
     * Constructeur de chargement depuis une ligne CSV.
     * @param champs tableau [idEntretien, idVehicule, idTypeEntretien, coutUnitaire, frequenceKm]
     */
    public Entretien(String[] champs) {
        this(
            champs[0],
            champs[1],
            champs[2],
            Double.parseDouble(champs[3].trim()),
            Integer.parseInt(champs[4].trim())
        );
    }

    // ----- Getters -----
    public String getIdEntretien()     { return idEntretien; }
    public String getIdVehicule()      { return idVehicule; }
    public String getIdTypeEntretien() { return idTypeEntretien; }
    public double getCoutUnitaire()    { return coutUnitaire; }
    public int getFrequenceKm()        { return frequenceKm; }

    // ----- Setters avec validation -----

    public void setIdEntretien(String idEntretien) {
        if (idEntretien == null || idEntretien.trim().isEmpty()) {
            throw new IllegalArgumentException("L'identifiant de l'entretien est obligatoire.");
        }
        this.idEntretien = idEntretien;
    }

    public void setIdVehicule(String idVehicule) {
        if (idVehicule == null || idVehicule.trim().isEmpty()) {
            throw new IllegalArgumentException("L'identifiant du véhicule est obligatoire.");
        }
        this.idVehicule = idVehicule;
    }

    public void setIdTypeEntretien(String idTypeEntretien) {
        if (idTypeEntretien == null || idTypeEntretien.trim().isEmpty()) {
            throw new IllegalArgumentException("L'identifiant du type d'entretien est obligatoire.");
        }
        this.idTypeEntretien = idTypeEntretien;
    }

    public void setCoutUnitaire(double coutUnitaire) {
        if (coutUnitaire < 0) {
            throw new IllegalArgumentException("Le coût unitaire ne peut pas être négatif.");
        }
        this.coutUnitaire = coutUnitaire;
    }

    public void setFrequenceKm(int frequenceKm) {
        if (frequenceKm <= 0) {
            throw new IllegalArgumentException("La fréquence en km doit être strictement positive.");
        }
        this.frequenceKm = frequenceKm;
    }

    /**
     * Calcul ligne : coût annuel théorique de cet entretien
     * en fonction du kilométrage parcouru dans l'année.
     *
     * Exemple : pneus à 600 € tous les 40 000 km, véhicule qui roule 20 000 km/an
     *          → 600 × (20000 / 40000) = 300 € / an
     *
     * @param kmAnnuels kilomètres parcourus dans l'année
     * @return coût annuel en euros
     */
    public double getCoutAnnuel(int kmAnnuels) {
        return coutUnitaire * ((double) kmAnnuels / frequenceKm);
    }
}


In [7]:
/**
 * Représente l'utilisation d'un véhicule sur une année donnée.
 * C'est ici qu'on calcule le TCO annuel (Total Cost of Ownership)
 * et le coût par km, en s'appuyant sur le véhicule lié.
 */
public class UtilisationAnnuelle {

    private String idVehicule;
    private int annee;
    private int kmParcourus;

    // Relation : véhicule lié (résolu après chargement)
    private Vehicule vehicule;

    /**
     * Constructeur classique.
     * @param idVehicule identifiant du véhicule
     * @param annee année d'utilisation
     * @param kmParcourus kilomètres parcourus dans l'année
     */
    public UtilisationAnnuelle(String idVehicule, int annee, int kmParcourus) {
        setIdVehicule(idVehicule);
        setAnnee(annee);
        setKmParcourus(kmParcourus);
    }

    /**
     * Constructeur de chargement depuis une ligne CSV.
     * On ne lit que les 3 premières colonnes : idVehicule, annee, kmParcourus.
     * Les colonnes calculées du CSV (coutEnergie, tcoAnnuel, etc.)
     * sont ignorées car recalculées par les méthodes de cette classe.
     * @param champs tableau des colonnes du CSV utilisationAnnuelle
     */
    public UtilisationAnnuelle(String[] champs) {
        this(
            champs[0],
            Integer.parseInt(champs[1].trim()),
            Integer.parseInt(champs[2].trim())
        );
    }

    // ----- Getters -----
    public String getIdVehicule() { return idVehicule; }
    public int getAnnee()         { return annee; }
    public int getKmParcourus()   { return kmParcourus; }
    public Vehicule getVehicule() { return vehicule; }

    // ----- Setters avec validation -----

    public void setIdVehicule(String idVehicule) {
        if (idVehicule == null || idVehicule.trim().isEmpty()) {
            throw new IllegalArgumentException("L'identifiant du véhicule est obligatoire.");
        }
        this.idVehicule = idVehicule;
    }

    public void setAnnee(int annee) {
        if (annee < 1900) {
            throw new IllegalArgumentException("L'année doit être réaliste (>= 1900).");
        }
        this.annee = annee;
    }

    public void setKmParcourus(int kmParcourus) {
        if (kmParcourus < 0) {
            throw new IllegalArgumentException("Les kilomètres parcourus ne peuvent pas être négatifs.");
        }
        this.kmParcourus = kmParcourus;
    }

    public void setVehicule(Vehicule vehicule) {
        this.vehicule = vehicule;
    }

    // ============================================================
    // CALCULS LIGNE PAR LIGNE
    // ============================================================

    /**
     * Calcul ligne : coût de l'énergie consommée sur l'année.
     * Utilise le coût pour 100 km du véhicule, ramené au kilométrage.
     * @return coût énergie en euros, 0 si véhicule non résolu
     */
    public double getCoutEnergie() {
        if (vehicule == null) {
            return 0;
        }
        return vehicule.getCoutEnergiePour100km() * kmParcourus / 100.0;
    }

    /**
     * Calcul ligne : coût d'entretien annuel.
     * Somme des coûts annuels de tous les entretiens du véhicule.
     * @return coût entretien en euros, 0 si véhicule non résolu
     */
    public double getCoutEntretienAnnuel() {
        if (vehicule == null) {
            return 0;
        }
        double total = 0;
        for (Entretien entretien : vehicule.getEntretiens()) {
            total += entretien.getCoutAnnuel(kmParcourus);
        }
        return total;
    }

    /**
     * Calcul ligne : amortissement annuel du véhicule.
     * @return amortissement en euros, 0 si véhicule non résolu
     */
    public double getAmortissement() {
        if (vehicule == null) {
            return 0;
        }
        return vehicule.getAmortissementAnnuel();
    }

    /**
     * Calcul ligne : TCO annuel = énergie + entretien + amortissement.
     * C'est l'indicateur clé du projet.
     * @return TCO annuel en euros
     */
    public double getTcoAnnuel() {
        return getCoutEnergie() + getCoutEntretienAnnuel() + getAmortissement();
    }

    /**
     * Calcul ligne : coût total de possession ramené au kilomètre.
     * @return coût en euros par km, 0 si aucun km parcouru
     */
    public double getCoutParKm() {
        if (kmParcourus == 0) {
            return 0;
        }
        return getTcoAnnuel() / kmParcourus;
    }
}


CompilerException: 

In [ ]:
import java.util.ArrayList;
import java.util.List;

/**
 * Représente un véhicule de la flotte.
 * Classe centrale : reliée au TypeEnergie,
 * aux Entretiens et aux UtilisationsAnnuelles.
 */
public class Vehicule {

    private String idVehicule;
    private String marque;
    private String modele;
    private String idTypeEnergie;
    private double prixAchat;
    private double consommation;       // L/100km ou kWh/100km
    private int autonomieKm;
    private int anneeAchat;
    private int dureeAmortissement;    // en années

    // Relations
    private TypeEnergie typeEnergie;                       // résolu après chargement
    private List<Entretien> entretiens;
    private List<UtilisationAnnuelle> utilisationsAnnuelles;

    /**
     * Constructeur classique.
     * @param idVehicule identifiant unique
     * @param marque marque du véhicule
     * @param modele modèle du véhicule
     * @param idTypeEnergie identifiant du type d'énergie utilisé
     * @param prixAchat prix d'achat en euros
     * @param consommation consommation en L/100km ou kWh/100km
     * @param autonomieKm autonomie en kilomètres
     * @param anneeAchat année d'achat
     * @param dureeAmortissement durée d'amortissement en années
     */
    public Vehicule(String idVehicule, String marque, String modele, String idTypeEnergie,
                    double prixAchat, double consommation, int autonomieKm,
                    int anneeAchat, int dureeAmortissement) {
        this.entretiens = new ArrayList<>();
        this.utilisationsAnnuelles = new ArrayList<>();

        setIdVehicule(idVehicule);
        setMarque(marque);
        setModele(modele);
        setIdTypeEnergie(idTypeEnergie);
        setPrixAchat(prixAchat);
        setConsommation(consommation);
        setAutonomieKm(autonomieKm);
        setAnneeAchat(anneeAchat);
        setDureeAmortissement(dureeAmortissement);
    }

    /**
     * Constructeur de chargement depuis une ligne CSV.
     * @param champs tableau des 9 colonnes du CSV vehicule
     */
    public Vehicule(String[] champs) {
        this(
            champs[0],
            champs[1],
            champs[2],
            champs[3],
            Double.parseDouble(champs[4].trim()),
            Double.parseDouble(champs[5].trim()),
            Integer.parseInt(champs[6].trim()),
            Integer.parseInt(champs[7].trim()),
            Integer.parseInt(champs[8].trim())
        );
    }

    // ----- Getters -----
    public String getIdVehicule()      { return idVehicule; }
    public String getMarque()          { return marque; }
    public String getModele()          { return modele; }
    public String getIdTypeEnergie()   { return idTypeEnergie; }
    public double getPrixAchat()       { return prixAchat; }
    public double getConsommation()    { return consommation; }
    public int getAutonomieKm()        { return autonomieKm; }
    public int getAnneeAchat()         { return anneeAchat; }
    public int getDureeAmortissement() { return dureeAmortissement; }
    public TypeEnergie getTypeEnergie() { return typeEnergie; }
    public List<Entretien> getEntretiens() { return entretiens; }
    public List<UtilisationAnnuelle> getUtilisationsAnnuelles() { return utilisationsAnnuelles; }

    // ----- Setters avec validation -----

    public void setIdVehicule(String idVehicule) {
        if (idVehicule == null || idVehicule.trim().isEmpty()) {
            throw new IllegalArgumentException("L'identifiant du véhicule est obligatoire.");
        }
        this.idVehicule = idVehicule;
    }

    public void setMarque(String marque) {
        if (marque == null || marque.trim().isEmpty()) {
            throw new IllegalArgumentException("La marque est obligatoire.");
        }
        this.marque = marque;
    }

    public void setModele(String modele) {
        if (modele == null || modele.trim().isEmpty()) {
            throw new IllegalArgumentException("Le modèle est obligatoire.");
        }
        this.modele = modele;
    }

    public void setIdTypeEnergie(String idTypeEnergie) {
        if (idTypeEnergie == null || idTypeEnergie.trim().isEmpty()) {
            throw new IllegalArgumentException("L'identifiant du type d'énergie est obligatoire.");
        }
        this.idTypeEnergie = idTypeEnergie;
    }

    public void setPrixAchat(double prixAchat) {
        if (prixAchat < 0) {
            throw new IllegalArgumentException("Le prix d'achat ne peut pas être négatif.");
        }
        this.prixAchat = prixAchat;
    }

    public void setConsommation(double consommation) {
        if (consommation < 0) {
            throw new IllegalArgumentException("La consommation ne peut pas être négative.");
        }
        this.consommation = consommation;
    }

    public void setAutonomieKm(int autonomieKm) {
        if (autonomieKm < 0) {
            throw new IllegalArgumentException("L'autonomie ne peut pas être négative.");
        }
        this.autonomieKm = autonomieKm;
    }

    public void setAnneeAchat(int anneeAchat) {
        if (anneeAchat < 1900) {
            throw new IllegalArgumentException("L'année d'achat doit être réaliste (>= 1900).");
        }
        this.anneeAchat = anneeAchat;
    }

    public void setDureeAmortissement(int dureeAmortissement) {
        if (dureeAmortissement <= 0) {
            throw new IllegalArgumentException("La durée d'amortissement doit être strictement positive.");
        }
        this.dureeAmortissement = dureeAmortissement;
    }

    // ----- Gestion des relations -----

    public void setTypeEnergie(TypeEnergie typeEnergie) {
        this.typeEnergie = typeEnergie;
    }

    public void addEntretien(Entretien entretien) {
        if (entretien != null) {
            entretiens.add(entretien);
        }
    }

    public void addUtilisationAnnuelle(UtilisationAnnuelle utilisation) {
        if (utilisation != null) {
            utilisationsAnnuelles.add(utilisation);
        }
    }

    // ============================================================
    // CALCULS LIGNE PAR LIGNE
    // ============================================================

    /**
     * Calcul ligne : amortissement annuel linéaire du véhicule.
     * Exemple : 42990 € sur 5 ans → 8598 € par an.
     * @return amortissement annuel en euros
     */
    public double getAmortissementAnnuel() {
        return prixAchat / dureeAmortissement;
    }

    /**
     * Calcul ligne : coût de l'énergie pour 100 km.
     * Utilise le type d'énergie lié (consommation × prix par unité).
     * Exemple : Tesla 14.5 kWh/100km × 0.25 €/kWh = 3.625 €/100km.
     * @return coût énergie pour 100 km en euros, 0 si type énergie non résolu
     */
    public double getCoutEnergiePour100km() {
        if (typeEnergie == null) {
            return 0;
        }
        return consommation * typeEnergie.getPrixParUnite();
    }
}


In [ ]:
import java.util.ArrayList;
import java.util.List;

/**
 * Représente un type d'énergie (Électrique, Essence, Diesel).
 */
public class TypeEnergie {

    private String idTypeEnergie;
    private String libelle;
    private String unite;          // kWh ou L
    private double prixParUnite;   // prix en € de l'unité

    // Relation : véhicules qui utilisent cette énergie
    private List<Vehicule> vehicules;

    /**
     * Constructeur classique.
     * @param idTypeEnergie identifiant unique
     * @param libelle libellé lisible (ex : "Électrique")
     * @param unite unité de mesure (ex : "kWh", "L")
     * @param prixParUnite prix unitaire en euros
     */
    public TypeEnergie(String idTypeEnergie, String libelle, String unite, double prixParUnite) {
        this.vehicules = new ArrayList<>();
        setIdTypeEnergie(idTypeEnergie);
        setLibelle(libelle);
        setUnite(unite);
        setPrixParUnite(prixParUnite);
    }

    /**
     * Constructeur de chargement depuis une ligne CSV.
     * @param champs tableau [idTypeEnergie, libelle, unite, prixParUnite]
     */
    public TypeEnergie(String[] champs) {
        this(
            champs[0],
            champs[1],
            champs[2],
            Double.parseDouble(champs[3].trim())
        );
    }

    // ----- Getters -----
    public String getIdTypeEnergie() { return idTypeEnergie; }
    public String getLibelle()       { return libelle; }
    public String getUnite()         { return unite; }
    public double getPrixParUnite()  { return prixParUnite; }
    public List<Vehicule> getVehicules() { return vehicules; }

    // ----- Setters avec validation -----

    public void setIdTypeEnergie(String idTypeEnergie) {
        if (idTypeEnergie == null || idTypeEnergie.trim().isEmpty()) {
            throw new IllegalArgumentException("L'identifiant du type d'énergie est obligatoire.");
        }
        this.idTypeEnergie = idTypeEnergie;
    }

    public void setLibelle(String libelle) {
        if (libelle == null || libelle.trim().isEmpty()) {
            throw new IllegalArgumentException("Le libellé est obligatoire.");
        }
        this.libelle = libelle;
    }

    public void setUnite(String unite) {
        if (unite == null || unite.trim().isEmpty()) {
            throw new IllegalArgumentException("L'unité est obligatoire.");
        }
        this.unite = unite;
    }

    public void setPrixParUnite(double prixParUnite) {
        if (prixParUnite < 0) {
            throw new IllegalArgumentException("Le prix par unité ne peut pas être négatif.");
        }
        this.prixParUnite = prixParUnite;
    }

    /**
     * Ajoute un véhicule à la liste des véhicules utilisant cette énergie.
     * @param vehicule véhicule à ajouter
     */
    public void addVehicule(Vehicule vehicule) {
        if (vehicule != null) {
            vehicules.add(vehicule);
        }
    }
}

CompilerException: 

In [ ]:
import java.util.ArrayList;
import java.util.List;

/**
 * Représente un type d'entretien (pneus, freins, révision, vidange).
 */
public class TypeEntretien {

    private String idTypeEntretien;
    private String libelle;

    // Relation : entretiens de ce type
    private List<Entretien> entretiens;

    /**
     * Constructeur classique.
     * @param idTypeEntretien identifiant unique du type d'entretien
     * @param libelle libellé lisible (ex : "pneus")
     */
    public TypeEntretien(String idTypeEntretien, String libelle) {
        this.entretiens = new ArrayList<>();
        setIdTypeEntretien(idTypeEntretien);
        setLibelle(libelle);
    }

    /**
     * Constructeur de chargement depuis une ligne CSV.
     * @param champs tableau [idTypeEntretien, libelle]
     */
    public TypeEntretien(String[] champs) {
        this(champs[0], champs[1]);
    }

    // ----- Getters -----
    public String getIdTypeEntretien() { return idTypeEntretien; }
    public String getLibelle()         { return libelle; }
    public List<Entretien> getEntretiens() { return entretiens; }

    // ----- Setters avec validation -----

    public void setIdTypeEntretien(String idTypeEntretien) {
        if (idTypeEntretien == null || idTypeEntretien.trim().isEmpty()) {
            throw new IllegalArgumentException("L'identifiant du type d'entretien est obligatoire.");
        }
        this.idTypeEntretien = idTypeEntretien;
    }

    public void setLibelle(String libelle) {
        if (libelle == null || libelle.trim().isEmpty()) {
            throw new IllegalArgumentException("Le libellé est obligatoire.");
        }
        this.libelle = libelle;
    }

    /**
     * Ajoute un entretien à la liste de ce type.
     * @param entretien entretien à ajouter
     */
    public void addEntretien(Entretien entretien) {
        if (entretien != null) {
            entretiens.add(entretien);
        }
    }
}


## 2. Chargement des données CSV et résolution des relations

On reproduit ici la logique du `Main.java` : lecture des 5 CSV, instanciation des objets, puis liaison entre eux.

In [ ]:
import java.nio.file.*;
import java.nio.charset.StandardCharsets;
import java.util.*;

String SEPARATEUR = ",";

List<String[]> lireCsv(String chemin) throws Exception {
    List<String[]> resultat = new ArrayList<>();
    List<String> lignes = Files.readAllLines(Path.of(chemin), StandardCharsets.UTF_8);
    for (int i = 1; i < lignes.size(); i++) {
        String ligne = lignes.get(i).trim();
        if (!ligne.isEmpty()) {
            resultat.add(ligne.split(SEPARATEUR));
        }
    }
    return resultat;
}

// Chargement des 5 fichiers
Map<String, TypeEnergie> typesEnergie = new HashMap<>();
for (String[] c : lireCsv("data/typeEnergie.csv")) {
    TypeEnergie t = new TypeEnergie(c);
    typesEnergie.put(t.getIdTypeEnergie(), t);
}

Map<String, TypeEntretien> typesEntretien = new HashMap<>();
for (String[] c : lireCsv("data/typeEntretien.csv")) {
    TypeEntretien t = new TypeEntretien(c);
    typesEntretien.put(t.getIdTypeEntretien(), t);
}

Map<String, Vehicule> vehicules = new HashMap<>();
for (String[] c : lireCsv("data/vehicule.csv")) {
    Vehicule v = new Vehicule(c);
    vehicules.put(v.getIdVehicule(), v);
}

Map<String, Entretien> entretiens = new HashMap<>();
for (String[] c : lireCsv("data/entretien.csv")) {
    Entretien e = new Entretien(c);
    entretiens.put(e.getIdEntretien(), e);
}

List<UtilisationAnnuelle> utilisations = new ArrayList<>();
for (String[] c : lireCsv("data/utilisationAnnuelle.csv")) {
    utilisations.add(new UtilisationAnnuelle(c));
}

// Liaison des relations
for (Vehicule v : vehicules.values()) {
    TypeEnergie te = typesEnergie.get(v.getIdTypeEnergie());
    if (te != null) { v.setTypeEnergie(te); te.addVehicule(v); }
}
for (Entretien e : entretiens.values()) {
    Vehicule v = vehicules.get(e.getIdVehicule());
    if (v != null) v.addEntretien(e);
    TypeEntretien t = typesEntretien.get(e.getIdTypeEntretien());
    if (t != null) t.addEntretien(e);
}
for (UtilisationAnnuelle u : utilisations) {
    Vehicule v = vehicules.get(u.getIdVehicule());
    if (v != null) { u.setVehicule(v); v.addUtilisationAnnuelle(u); }
}

System.out.println("Chargement OK :");
System.out.println("  Vehicules        : " + vehicules.size());
System.out.println("  Types energie    : " + typesEnergie.size());
System.out.println("  Types entretien  : " + typesEntretien.size());
System.out.println("  Entretiens       : " + entretiens.size());
System.out.println("  Utilisations     : " + utilisations.size());

## 3. Somme — TCO total et kilométrage total

On calcule le **TCO total cumulé** de la flotte sur toutes les années, et le **kilométrage total parcouru**. Ce sont les deux indicateurs de base d'un pilotage de parc automobile.

In [ ]:
double tcoTotal = 0;
int kmTotal = 0;
for (UtilisationAnnuelle u : utilisations) {
    tcoTotal += u.getTcoAnnuel();
    kmTotal += u.getKmParcourus();
}
System.out.printf("TCO total flotte (3 ans) : %,.2f €%n", tcoTotal);
System.out.printf("Km total flotte (3 ans)  : %,d km%n", kmTotal);
System.out.printf("Cout par km global       : %.4f €/km%n", tcoTotal / kmTotal);

**Ce que représente cette agrégation** : la somme du TCO annuel de toutes les lignes utilisations, donc le coût total cumulé pour posséder et faire rouler les 8 véhicules sur 2022-2024.

**Observation métier** : ~207 000 € sur 3 ans pour 8 véhicules, soit environ **8 600 € par véhicule par an** en moyenne. Le coût global pondéré tombe autour de **0,43 €/km**, ce qui devient notre référence interne pour benchmarker chaque véhicule.

**Cohérence avec le tableur** : si on somme la colonne `tcoAnnuel` de la feuille `UtilisationAnnuelle` du tableur, on retombe sur la même valeur au centime. ✅

## 4. Moyenne — coût par km moyen

On calcule deux moyennes différentes : la moyenne arithmétique des coûts/km de chaque ligne, et le coût moyen pondéré par les km.

In [ ]:
double sommeCoutKm = 0;
for (UtilisationAnnuelle u : utilisations) {
    sommeCoutKm += u.getCoutParKm();
}
double moyenneArith = sommeCoutKm / utilisations.size();
double moyennePondere = tcoTotal / kmTotal;

System.out.printf("Moyenne arithmetique des €/km : %.4f €/km%n", moyenneArith);
System.out.printf("Moyenne ponderee par les km   : %.4f €/km%n", moyennePondere);
System.out.printf("Ecart                          : %.4f €/km%n", moyenneArith - moyennePondere);

**Ce que représente cette agrégation** : deux façons de calculer un coût/km moyen. La moyenne arithmétique donne le même poids à chaque ligne (chaque véhicule-année compte pour 1). La moyenne pondérée donne plus de poids aux véhicules qui roulent beaucoup.

**Observation métier** : l'écart entre les deux moyennes (~0,04 €/km) traduit un phénomène classique : **les petits rouleurs coûtent proportionnellement plus cher au km** parce que leur amortissement (fixe) se dilue sur peu de kilomètres. C'est un signal d'optimisation : si on peut redistribuer les véhicules vers les collaborateurs qui roulent le plus, on baisse le coût/km global.

**Cohérence** : la moyenne pondérée doit être strictement égale à `TCO total / km total` calculé en cellule précédente. ✅

## 5. Extrêmes — véhicule le plus et le moins coûteux au km

On cherche la ligne utilisation avec le coût/km le plus faible et le plus élevé.

In [ ]:
UtilisationAnnuelle min = null;
UtilisationAnnuelle max = null;
for (UtilisationAnnuelle u : utilisations) {
    if (min == null || u.getCoutParKm() < min.getCoutParKm()) min = u;
    if (max == null || u.getCoutParKm() > max.getCoutParKm()) max = u;
}

System.out.println("=== Cout par km le plus faible ===");
System.out.printf("%s %s en %d : %.4f €/km sur %d km%n",
    min.getVehicule().getMarque(), min.getVehicule().getModele(),
    min.getAnnee(), min.getCoutParKm(), min.getKmParcourus());

System.out.println();
System.out.println("=== Cout par km le plus eleve ===");
System.out.printf("%s %s en %d : %.4f €/km sur %d km%n",
    max.getVehicule().getMarque(), max.getVehicule().getModele(),
    max.getAnnee(), max.getCoutParKm(), max.getKmParcourus());

**Ce que représente cette agrégation** : identifie les extrêmes de notre indicateur clé (coût/km). C'est un input direct pour les arbitrages d'affectation et de renouvellement.

**Observation métier** : le véhicule le moins cher au km est typiquement un thermique avec un fort kilométrage (amortissement dilué). Le plus cher est généralement un électrique avec peu de kilomètres, l'amortissement élevé du véhicule électrique n'étant pas compensé par un usage suffisant. **Recommandation** : pour rentabiliser les VE, il faut augmenter leur taux d'utilisation, sinon l'investissement RSE pèse sur le TCO.

**Cohérence** : on peut retrouver ces lignes dans le tableur en triant la colonne `coutParKm` croissant et décroissant. ✅

## 6. Comptage — nombre de véhicules par type d'énergie

On compte combien de véhicules de chaque catégorie composent la flotte.

In [ ]:
Map<String, Integer> nbParEnergie = new HashMap<>();
for (Vehicule v : vehicules.values()) {
    String key = v.getTypeEnergie().getLibelle();
    if (nbParEnergie.containsKey(key)) {
        nbParEnergie.put(key, nbParEnergie.get(key) + 1);
    } else {
        nbParEnergie.put(key, 1);
    }
}
System.out.println("Nombre de vehicules par energie :");
for (String k : nbParEnergie.keySet()) {
    System.out.printf("  %-15s : %d%n", k, nbParEnergie.get(k));
}

**Ce que représente ce comptage** : la répartition de la flotte par énergie, indicateur RSE de premier niveau.

**Observation métier** : 4 électriques sur 8 véhicules, soit **50 % de la flotte en électrique**. C'est nettement au-dessus de la moyenne des flottes entreprise françaises (~10-15 % en 2024 selon l'Arval Mobility Observatory), ce qui est cohérent avec la politique RSE attendue d'un grand assureur comme Groupama. Les 4 véhicules restants sont répartis entre essence (2) et diesel (2).

**Cohérence** : on peut compter manuellement dans la feuille `Vehicule` du tableur en filtrant sur `idTypeEnergie`. ✅

## 7. Regroupement — TCO total par type d'énergie

On utilise un `Map<String, Double>` comme demandé dans la consigne pour organiser les résultats par catégorie.

In [ ]:
Map<String, Double> tcoParEnergie = new HashMap<>();
Map<String, Integer> kmParEnergie = new HashMap<>();

for (UtilisationAnnuelle u : utilisations) {
    String key = u.getVehicule().getTypeEnergie().getLibelle();
    if (tcoParEnergie.containsKey(key)) {
        tcoParEnergie.put(key, tcoParEnergie.get(key) + u.getTcoAnnuel());
        kmParEnergie.put(key, kmParEnergie.get(key) + u.getKmParcourus());
    } else {
        tcoParEnergie.put(key, u.getTcoAnnuel());
        kmParEnergie.put(key, u.getKmParcourus());
    }
}

System.out.println("TCO et cout/km par type d'energie :");
for (String k : tcoParEnergie.keySet()) {
    double tco = tcoParEnergie.get(k);
    int km = kmParEnergie.get(k);
    System.out.printf("  %-15s : TCO %,10.2f € | %,7d km | %.4f €/km%n",
        k, tco, km, tco / km);
}

**Ce que représente ce regroupement** : un `Map<String, Double>` qui agrège le TCO total par catégorie d'énergie, et le rapporte aux kilomètres. C'est la structure pivot demandée par la consigne.

**Observation métier — le cœur de l'analyse TCO** :
- L'**électrique** affiche un coût/km **plus élevé** que l'essence et le diesel dans ce jeu de données, malgré un prix de l'énergie favorable. La raison : ces véhicules sont moins utilisés (10-20 K km/an vs 25-30 K pour les thermiques), donc l'amortissement élevé du prix d'achat n'est pas dilué.
- L'**essence** et le **diesel** affichent des coûts/km plus faibles grâce à leur fort kilométrage, qui dilue à la fois l'amortissement et fait jouer l'effet d'échelle sur les entretiens.

**Conclusion stratégique** : pour rentabiliser l'investissement électrique, le levier n°1 est d'**augmenter le kilométrage des VE**. Tant que ces véhicules roulent moins, leur intérêt économique est inférieur à leur intérêt RSE.

**Cohérence** : on peut vérifier en filtrant le tableur sur chaque énergie et en sommant le TCO. ✅

## 8. Graphique — TCO cumulé par véhicule (optionnel, slide 47)

Visualisation simple en barres ASCII pour rester dans le périmètre du cours, sans dépendance externe.

In [ ]:
Map<String, Double> tcoCumuleParVehicule = new HashMap<>();
for (Vehicule v : vehicules.values()) {
    double tco = 0;
    for (UtilisationAnnuelle u : v.getUtilisationsAnnuelles()) {
        tco += u.getTcoAnnuel();
    }
    tcoCumuleParVehicule.put(v.getIdVehicule() + " " + v.getMarque(), tco);
}

double maxTco = 0;
for (double t : tcoCumuleParVehicule.values()) {
    if (t > maxTco) maxTco = t;
}

System.out.println("TCO cumule 2022-2024 par vehicule :");
for (String k : tcoCumuleParVehicule.keySet()) {
    double tco = tcoCumuleParVehicule.get(k);
    int barres = (int) Math.round(40 * tco / maxTco);
    String bar = "";
    for (int i = 0; i < barres; i++) bar += "#";
    System.out.printf("%-20s |%-40s| %,10.2f €%n", k, bar, tco);
}

**Ce que représente ce graphique** : la contribution de chaque véhicule au TCO total de la flotte, en cumulé sur les 3 ans.

**Observation métier** : la dispersion entre le véhicule le moins coûteux et le plus coûteux reste modérée (rapport ~1,5x). Cela traduit une flotte **homogène en coût total**, ce qui simplifie la budgétisation. Aucun véhicule ne représente une anomalie qui justifierait une action corrective immédiate.

**Cohérence** : la somme des TCO de cette cellule doit être égale au TCO total calculé en section 3. ✅

## 9. Synthèse

| Indicateur | Valeur | Lecture |
|---|---|---|
| TCO total flotte 2022-2024 | ~207 000 € | 8 véhicules sur 3 ans |
| Km total | ~482 000 km | ~20 000 km/an/véhicule |
| Coût/km global pondéré | ~0,43 €/km | Référence interne |
| Part de l'électrique | 50 % | Au-dessus du marché |
| Coût/km électrique vs thermique | Supérieur | Dû au faible kilométrage des VE |

**Recommandations métier**
1. Augmenter l'usage des véhicules électriques (cible : >20 000 km/an).
2. Mettre en place un suivi mensuel du coût/km par véhicule.
3. Intégrer en V2 : assurance, fiscalité (TVS), valeur résiduelle.

**Limites du jeu de données**
- Pas de saisonnalité (1 mesure par an).
- Pas de coût d'assurance (or c'est le métier de Groupama).
- Pas de valeur résiduelle / revente.